# 🔴 MODELO 6: LSTM + Early Stopping (Control automático)

## 📋 Objetivo del ejercicio
Usar **Early Stopping** para parar el entrenamiento automáticamente.

## ⚠️ Problema común
- No sabemos cuántas épocas son óptimas
- Pocas épocas → underfitting
- Muchas épocas → overfitting
- Tener que vigilar manualmente el training

## ✅ Solución: Early Stopping
**Callback** que para automáticamente cuando una métrica deja de mejorar.

### 📚 Teoría de examen:
> "Early Stopping es un callback que monitorea una métrica (ej: val_loss)
> y para el entrenamiento si no mejora por N épocas (patience).
> Además, puede restaurar los mejores pesos automáticamente."

### 🔑 Parámetros importantes:
- **monitor**: Métrica a monitorear ('val_loss', 'val_accuracy')
- **patience**: Épocas a esperar sin mejora antes de parar
- **restore_best_weights**: Si True, restaura pesos de la mejor época
- **mode**: 'min' (para loss), 'max' (para accuracy), 'auto'

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)

In [ ]:
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(test['message'].values), maxlen=MAX_LEN, padding='post')

print("✅ Datos preparados")

## 🏗️ Modelo LSTM con Early Stopping

### 📚 Configuración de Early Stopping

```python
EarlyStopping(
    monitor='val_loss',          # Métrica a observar
    patience=3,                  # Épocas sin mejora antes de parar
    restore_best_weights=True,   # Restaurar mejores pesos
    mode='min',                  # 'min' para loss, 'max' para accuracy
    verbose=1                    # Mostrar mensaje cuando pare
)
```

### 🔑 Pregunta de examen típica:
**P: ¿Qué hace restore_best_weights=True?**

**R: Cuando para el entrenamiento, restaura los pesos de la época con mejor val_loss,
no los de la última época. Así no te quedas con un modelo sobreajustado.**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.4),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## ⚡ Configurar Early Stopping

### Opciones comunes:

**1. Monitorear val_loss (más común):**
```python
EarlyStopping(monitor='val_loss', patience=3, mode='min')
```

**2. Monitorear val_accuracy:**
```python
EarlyStopping(monitor='val_accuracy', patience=3, mode='max')
```

**3. Con más paciencia (dataset pequeño):**
```python
EarlyStopping(monitor='val_loss', patience=5)
```

**4. Agresivo (detección rápida):**
```python
EarlyStopping(monitor='val_loss', patience=2)
```

In [ ]:
# ✨ Configurar Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',           # Observar pérdida de validación
    patience=3,                   # Esperar 3 épocas sin mejora
    restore_best_weights=True,    # Restaurar mejores pesos
    verbose=1                     # Mostrar mensaje
)

print("✅ Early Stopping configurado")
print("   - Monitor: val_loss")
print("   - Patience: 3 épocas")
print("   - Restaurar mejores pesos: Sí")

In [ ]:
# Entrenar con MUCHAS épocas - Early Stopping parará cuando sea necesario
print("🚀 Entrenando con Early Stopping...")
print("   Configuradas 50 épocas, pero parará antes si no mejora\n")

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=50,  # ⚡ Muchas épocas, pero Early Stopping parará antes
    batch_size=32,
    callbacks=[early_stop],  # ✨ Añadir callback
    verbose=1
)

print(f"\n✅ Entrenamiento parado en época: {len(history.history['loss'])}")
print(f"   (de 50 configuradas)")

### 📊 Observa:
- El entrenamiento se detuvo **antes** de las 50 épocas
- Keras mostró un mensaje: "Restoring model weights from the end of the best epoch"
- Esto significa que detectó overfitting y paró automáticamente

In [ ]:
import matplotlib.pyplot as plt

# Visualizar dónde paró
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
plt.axvline(x=len(history.history['loss'])-1, color='r', linestyle='--', label='Paró aquí')
plt.title('Accuracy - Early Stopping en acción')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s')
plt.axvline(x=len(history.history['loss'])-1, color='r', linestyle='--', label='Paró aquí')
# Marcar mejor época (3 épocas antes del final por patience=3)
best_epoch = len(history.history['loss']) - 4
if best_epoch >= 0:
    plt.axvline(x=best_epoch, color='g', linestyle=':', label='Mejor época')
plt.title('Loss - Early Stopping en acción')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📌 La línea verde es la MEJOR época (pesos restaurados)")
print("📌 La línea roja es donde PARÓ el entrenamiento")

In [ ]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")
print(f"\n✅ Estos son los resultados de la MEJOR época, no la última")

In [ ]:
# Análisis del Early Stopping
total_epochs = len(history.history['loss'])
print(f"\n📊 ANÁLISIS DEL EARLY STOPPING:")
print(f"   - Épocas configuradas: 50")
print(f"   - Épocas ejecutadas: {total_epochs}")
print(f"   - Épocas ahorradas: {50 - total_epochs}")
print(f"   - Tiempo ahorrado: ~{((50 - total_epochs) / 50) * 100:.1f}%")
print(f"\n💡 Early Stopping previno {50 - total_epochs} épocas innecesarias!")

In [ ]:
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_early_stopping.csv', index=False)

print("✅ Submission creado: submission_early_stopping.csv")

---

## 📚 Resumen para Examen

### ✅ Ventajas Early Stopping:
- **Previene overfitting automáticamente**
- **Ahorra tiempo** (no entrenas épocas innecesarias)
- **No necesitas adivinar** el número óptimo de épocas
- **Restaura mejores pesos** automáticamente
- No añade parámetros al modelo

### ⚙️ Configuración típica:
```python
EarlyStopping(
    monitor='val_loss',           # Métrica a observar
    patience=3,                   # Épocas a esperar
    restore_best_weights=True,    # Restaurar mejores pesos
    verbose=1                     # Mostrar mensajes
)
```

### 🎯 Valores de patience típicos:
- **patience=2**: Agresivo, para rápido
- **patience=3-5**: Estándar, buen balance
- **patience=7-10**: Conservador, datasets con mucho ruido

### 🔑 Preguntas de examen:

**P1: ¿Qué hace restore_best_weights=True?**
> R: Restaura los pesos de la época con mejor métrica monitoreada, no los de la última época.

**P2: ¿Cuándo usar patience=5 vs patience=2?**
> R: patience=5 para datasets pequeños o con ruido (más paciencia). 
> patience=2 para datasets grandes y limpios (detección rápida).

**P3: ¿Se puede usar con cualquier métrica?**
> R: Sí. val_loss (mode='min'), val_accuracy (mode='max'), 
> o cualquier métrica custom.

### 💡 Buena práctica:
> "Siempre usa Early Stopping con restore_best_weights=True.
> Configura muchas épocas (ej: 50-100) y deja que pare solo."